# Differentiable Monte Carlo — The Real Deal

**Goal:** Convert the paper's MC simulation into a fully differentiable pipeline in PyTorch.

**Plan (step by step):**
1. Implement the full MC simulation as a PyTorch module
2. Make the sampling differentiable (from toy-experiment-sampling)
3. Make the Voigt fit differentiable via implicit differentiation (from toy-experiment-fitting)
4. Compare simulated vs experimental distributions using smooth density comparison
5. Optimize (γ, n̄) via gradient descent

---

## Step 1: Imports & Setup

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import dataclass

print('Ready')

## Step 2: One MC Run (Continuous)

One run = simulate one PLE scan with **continuous photon positions** (no binning).

Steps:
1. Sample total photon count $n \sim \mathcal{N}(\bar{n}, \sigma)$
2. Draw $n$ photon frequencies from the Cauchy($\gamma$) lineshape (continuous)
3. Add background noise: draw $n_{\text{bg}} \sim \text{Poisson}(\lambda)$, distribute uniformly
4. **Return:** all photon frequencies concatenated into one tensor

Later: fit a Voigt to these continuous samples to extract one linewidth $w_i$.

In [ ]:
@dataclass
class MCParams:
    gamma: float       # HWHM of Cauchy (MHz) — optimized
    nbar: float        # mean photon count — optimized
    sigma: float = 6.0  # noise std (fixed, from paper)
    lambda_: float = 2.0  # mean background counts (fixed, from paper)

# Frequency window (from paper)
FREQ_MIN = -75.0   # MHz
FREQ_MAX = 75.0    # MHz
print(f'Frequency window: [{FREQ_MIN}, {FREQ_MAX}] MHz')

## Step 2: Separated Sampling Functions

In [ ]:
def sample_n(params, epsilon):
    """Sample photon count N ~ N(nbar, sigma), reparameterized."""
    n_float = params.nbar + params.sigma * epsilon
    return max(round(n_float), 0)

def sample_cauchy(n, gamma):
    """Sample n photon frequencies from Cauchy(gamma), clipped to freq window."""
    if n == 0:
        return torch.tensor([], dtype=torch.float32)
    u = np.random.uniform(0, 1, n)
    samples = gamma * np.tan(np.pi * (u - 0.5))
    return torch.tensor(np.clip(samples, FREQ_MIN, FREQ_MAX), dtype=torch.float32)

def sample_background(lambda_):
    """Sample background events uniformly across freq window."""
    n_bg = np.random.poisson(lambda_)
    if n_bg == 0:
        return torch.tensor([], dtype=torch.float32)
    bg = np.random.uniform(FREQ_MIN, FREQ_MAX, n_bg)
    return torch.tensor(bg, dtype=torch.float32)

## Step 3: Full Run — Params In, FWHM Out

In [ ]:
def full_run(params, epsilon):
    """One full MC run: sample N → Cauchy photons → background → fit → FWHM."""
    n = sample_n(params, epsilon)
    signal = sample_cauchy(n, params.gamma)
    bg = sample_background(params.lambda_)
    
    photons = torch.cat([signal, bg])
    if len(photons) < 3:
        return 50.0  # too few photons, return safe default
    
    fwhm, _ = fit_pseudo_voigt(photons)
    return fwhm


# Quick test
params = MCParams(gamma=15.0, nbar=40.0)
np.random.seed(42)
eps = np.random.normal()
w = full_run(params, eps)
print(f'Full run: params gamma={params.gamma}, nbar={params.nbar}')
print(f'  epsilon = {eps:.2f}')
print(f'  extracted FWHM = {w:.2f} MHz')

## Step 4: Fitting a Voigt to the Photon Point Cloud

(Unchanged — `fit_pseudo_voigt` used by `full_run`)

In [ ]:
def pseudo_voigt_log_pdf(freqs, center, log_gamma, log_sigma_g, logit_eta):
    """Log-PDF of pseudo-Voigt. Params in unconstrained space for stable opt."""
    gamma = torch.exp(log_gamma)
    sigma_g = torch.exp(log_sigma_g)
    eta = torch.sigmoid(logit_eta)
    
    # Gaussian part
    gauss = torch.exp(-0.5 * ((freqs - center) / sigma_g) ** 2)
    gauss = gauss / (sigma_g * torch.sqrt(torch.tensor(2.0 * torch.pi)))
    
    # Lorentzian part
    lorentz = (gamma / torch.pi) / ((freqs - center) ** 2 + gamma ** 2)
    
    pdf = eta * gauss + (1 - eta) * lorentz
    return torch.log(pdf + 1e-30)

def fit_pseudo_voigt(photons, n_iters=200):
    """
    Fit pseudo-Voigt to photon frequencies via MLE (L-BFGS).
    Returns (fwhm, params_dict).
    """
    if len(photons) < 3:
        return 50.0, None
    
    freqs = photons.clone().detach().float()
    
    center = torch.tensor(float(freqs.median()), requires_grad=True)
    log_gamma = torch.tensor(np.log(15.0), requires_grad=True)
    log_sigma_g = torch.tensor(np.log(5.0), requires_grad=True)
    logit_eta = torch.tensor(0.0, requires_grad=True)
    
    optimizer = torch.optim.LBFGS([center, log_gamma, log_sigma_g, logit_eta],
                                   max_iter=n_iters, line_search_fn='strong_wolfe')
    
    def closure():
        optimizer.zero_grad()
        log_pdf = pseudo_voigt_log_pdf(freqs, center, log_gamma, log_sigma_g, logit_eta)
        nll = -log_pdf.mean()
        nll.backward()
        return nll
    
    optimizer.step(closure)
    
    gamma = torch.exp(log_gamma).item()
    fwhm = 2.0 * gamma
    
    return fwhm, {
        'center': center.item(),
        'gamma': gamma,
        'sigma_g': torch.exp(log_sigma_g).item(),
        'eta': torch.sigmoid(logit_eta).item(),
        'fwhm': fwhm,
    }


# Test
torch.manual_seed(42)
true_gamma = 15.0
u = torch.rand(200)
true_photons = true_gamma * torch.tan(torch.pi * (u - 0.5))
true_photons = torch.clamp(true_photons, FREQ_MIN, FREQ_MAX)

fwhm, p = fit_pseudo_voigt(true_photons)
print(f'True gamma = {true_gamma} MHz  →  true FWHM = {2*true_gamma} MHz')
print(f'Fit  gamma = {p["gamma"]:.2f} MHz  →  fit FWHM = {fwhm:.2f} MHz')
print(f'Fit: center={p["center"]:.2f}, sigma_g={p["sigma_g"]:.2f}, eta={p["eta"]:.2f}')